In [ ]:
#imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from scipy.interpolate import interp1d
#import mse
from sklearn.metrics import mean_squared_error
from scipy.signal import medfilt


In [ ]:
participant_list = [6,7,8,9,10,11,28,29,44,45,50,51,52,53,64,65,76,77,78,79,80,81,82,83] #to update

In [ ]:
#to get average duration of the videos (cortesy of chatgpt)

import os
import moviepy
from moviepy.editor import VideoFileClip
import numpy as np

# Set the root directory to search for videos
root_directory = "../[Study Data]/used_corae/"

# Initialize a list to store the durations of videos
durations = []

# Loop through all subdirectories and find videos without "merged" in their names
for subdir, dirs, files in os.walk(root_directory):
    for file in files:
        # Check if file is an mp4 video and does not contain "merged" in its name
        if file.endswith(".mov") and "merged" not in file:
            # Get the full path to the video file
            video_path = os.path.join(subdir, file)
            # Load the video file using moviepy
            video = VideoFileClip(video_path)
            # Get the duration of the video in seconds
            duration = video.duration
            # Add the duration to the list of durations
            durations.append(duration)
            # Close the video file
            video.close()

# Calculate the average and standard deviation of the durations
avg_duration = np.mean(durations)
std_duration = np.std(durations)

# Print the average and standard deviation of the durations
print(f"Average video duration: {avg_duration:.2f} seconds")
print(f"Standard deviation of video duration: {std_duration:.2f} seconds")
#print total number of videos and of durations (sum of durations)
print(f"Total number of videos: {len(durations)}")
print(f"Total number of durations: {sum(durations)}")

# CORAE DATA

In [ ]:
#one by one, open json files in a folder, read them and add them to pandas dataframe as two columns: 'time' and 'value'

#see files in folder
filelist = os.listdir('../[Study Data]/ParticipantsJSON')
filelist = [file for file in filelist if file.endswith('.json')]

#make empty dataframe
df = pd.DataFrame(columns=['participant_number','session','timestamp', 'frame_number','frame_seconds','seconds', 'value'], index=(range(len(participant_list))))
dfrow = 0
#open files one by one and read
for file in filelist:
    pnumber = int(file[:2])
    print(pnumber)
    if pnumber in participant_list:
        
        #session number, knowing it started at 0 and it's 2 participants per session (p number start at 0)
        session = np.ceil((pnumber-1)/2)

        # Opening JSON file
        f = open('../[Study Data]/ParticipantsJSON/' + file)
        
        # returns JSON object as 
        # a dictionary
        data = json.load(f)
        #print(data)
    
        #adding to pandas df, one column for time==keys and one for values, but a single row
        #get dictionary keys
        keys = list(data.keys())
        #get dictionary values
        values_list = list(data.values())
        print(len(keys))
        values = []
        #turn every value into an int
        for value in values_list:
            value = int(value)
            values.append(value)
            
        if keys[0] == "ID":
            keys = keys[1:]
            values = values[1:]
        frames = []
        seconds = []
        frameseconds = []
        for keyf in keys:
            #split keyf according to ":"
            keyf = keyf.split(":")
            #get frames (last)
            frames.append(int(keyf[-1]))
            frameseconds.append(int(keyf[-1])/30)
            #get seconds (second to last all the way to the first)
            seconds.append(int(keyf[-2]) + 60 * int(keyf[-3]) + 3600 * int(keyf[-4]))

            
        #place in df
        df.loc[dfrow] = [pnumber,session, keys, frames,frameseconds,seconds,values]
        dfrow += 1

        # Closing file
        f.close()

df = df.sort_values(by=['participant_number'])
df.index = range(len(df))
#save df as csv
df.to_csv('./participant_json.csv', index=False)
df

In [ ]:
#explore granularity of the results by looking at frame_seconds per participant (time intervals)

datapoints = []
intervals = []

for ind, row in df.iterrows():
    timestamps = df['frame_seconds'][ind]
    #see interval between timestamps
    
    for i in range(len(timestamps)-1):
        intervals.append(timestamps[i+1]-timestamps[i])
    
    datapoints.append(len(timestamps))


#plot
plt.figure(figsize=(10,5))
plt.hist(intervals, bins=100)
plt.title('Distribution of time intervals between frames')
plt.xlabel('Time interval (seconds)')
plt.ylabel('Frequency')
plt.show()

#print
print('Average time interval between frames: ', np.mean(intervals))
print('Median time interval between frames: ', np.median(intervals))
print('Standard deviation of time interval between frames: ', np.std(intervals))
print('Number of datapoints: ', len(intervals))

#average datapoints per participant
print('Average datapoints per participant: ', np.mean(datapoints))
print('Median datapoints per participant: ', np.median(datapoints))
print('Standard deviation of datapoints per participant: ', np.std(datapoints))





In [ ]:
#now, the same for values

datapoints = []
value_range_pp = []
value_avg_pp = []
value_std_pp = []

for ind, row in df.iterrows():

    values = df['value'][ind]
    #see range of values
    value_range_pp.append(max(values)-min(values))
    value_avg_pp.append(np.mean(values))
    value_std_pp.append(np.std(values))
    datapoints.extend(values)

    print('Participant ', df['participant_number'][ind], ' has range of values: ', max(values)-min(values))
    print('Participant ', df['participant_number'][ind], ' has average of values: ', np.mean(values))
#print average value overall
print('Average value overall: ', np.mean(datapoints))
print('Median value overall: ', np.median(datapoints))
print('Standard deviation of value overall: ', np.std(datapoints))
print('Range of value overall: ', max(datapoints)-min(datapoints))
print('Average range of values per participant: ', np.mean(value_range_pp))
print('Median range of values per participant: ', np.median(value_range_pp))
print('Standard deviation of range of values per participant: ', np.std(value_range_pp))


#plot
plt.figure(figsize=(10,5))
plt.hist(datapoints, bins=100)
plt.title('Distribution of values')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.show()

#plot range of values per participant
plt.figure(figsize=(10,5))
plt.hist(value_range_pp, bins=100)
plt.title('Distribution of range of values per participant')
plt.xlabel('Range of values')
plt.ylabel('Frequency')
plt.show()

#plot average of values per participant
plt.figure(figsize=(10,5))
plt.hist(value_avg_pp, bins=100)
plt.title('Distribution of average of values per participant')
plt.xlabel('Average of values')
plt.ylabel('Frequency')
plt.show()

#plot standard deviation of values per participant
plt.figure(figsize=(10,5))
plt.hist(value_std_pp, bins=100)
plt.title('Distribution of standard deviation of values per participant')
plt.xlabel('Standard deviation of values')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#per session, plot frames_seconds vs values
#sessions
sessions = df['session'].unique()
for session in sessions:
    #get df
    df_session = df[df['session']==session]
    #plot for one partipant, then plot the other
    plt.figure(figsize=(10,5))
    for participant in df_session['participant_number'].unique():
        df_participant = df_session[df_session['participant_number']==participant]
        x = np.array(df_participant['frame_seconds'].values[0])
        y = np.array(df_participant['value'].values[0])
        plt.plot(x,y,label = 'Participant ' + str(participant))
    plt.title('Session ' + str(session))
    plt.legend()
    plt.xlabel('Frame seconds')
    plt.ylabel('Value')
    plt.show()

In [ ]:
#now, plot the integral of the values over time, subtracting negative values
#sessions
sessions = df['session'].unique()
for session in sessions:
    #get df
    df_session = df[df['session']==session]
    #plot for one partipant, then plot the other
    plt.figure(figsize=(10,5))
    #get the integral of the values
    for participant in df_session['participant_number'].unique():
        df_participant = df_session[df_session['participant_number']==participant]
        x = np.array(df_participant['frame_seconds'].values[0])
        y = np.array(df_participant['value'].values[0])
        ysum = np.cumsum(y)
        #plot, with filled area under the line until 0 or over the line if ysum is negative
        plt.plot(x,ysum,label = 'Participant ' + str(participant))
        plt.fill_between(x, ysum, where=ysum>=0, interpolate=True, alpha=0.5)
        plt.fill_between(x, ysum, where=ysum<=0, interpolate=True, alpha=0.5)
        #plt.plot(x,ysum,label = 'Participant ' + str(participant))
    plt.title('Session ' + str(session))
    plt.legend()
    plt.xlabel('Frame seconds')
    plt.ylabel('Cumulative Interpersonal Perception')
    plt.show()

In [ ]:
#for each session, calculate degree of disagreement between participants with a scale of 0.1 seconds

#create new df with columns: session, participant, frame_seconds, value
df_interp = pd.DataFrame(columns=['session','participant_number','frame_seconds','value'], index=range(len(df)))

#first, get session df
sessions = df['session'].unique()
for session in sessions:
    df_session = df[df['session']==session]
    #get all participants
    participants = df_session['participant_number'].unique()
    #per participant, get all frame_seconds
    for participant in participants:
        df_p = df_session[df_session['participant_number']==participant]
  
        frame_seconds = df_p['frame_seconds'].values[0]
        values = df_p['value'].values[0]
        #now, interpolate both arrays to have the frames_seconds as the same at a 0.1 second interval
        #get min and max frame_seconds
        min_frame_seconds = min(frame_seconds)
        max_frame_seconds = np.round(max(frame_seconds),1) 
        
        #create new frame_seconds array
        new_frame_seconds = np.arange(0.0,max_frame_seconds,0.1)
        
        #interpolate values (step interpolation)
        #Create the interpolation function
        f = interp1d(frame_seconds, values, kind='next',fill_value='extrapolate')
        # Use the interpolation function to get the corresponding y-values
        new_values = f(new_frame_seconds)

        #print('new_session')
        #print(len(new_frame_seconds),len(new_values))
        #print(len(frame_seconds),len(values))
        
        #add to new df
        df_interp.loc[df_p.index] = [session,participant,new_frame_seconds,new_values]

    #print(df_interp.head())

    #just to check interpolation:
    ##plot for one partipant, then plot the other in the session
    #plt.figure(figsize=(10,5))
    ##plot from values in df_interp
    #for p in [participant -1, participant]:
    #    df_p = df_interp[df_interp['participant_number']==p]
    #    x = np.array(df_p['frame_seconds'].values[0])
    #    y = np.array(df_p['value'].values[0])
    #    plt.plot(x,y,label = 'Participant ' + str(p))
    #    #also plot the original values
    #    df_p = df_session[df_session['participant_number']==p]
    #    x = np.array(df_p['frame_seconds'].values[0])
    #    y = np.array(df_p['value'].values[0])
    #    plt.plot(x,y,label = 'Participant ' + str(p) + ' original')

    #plt.title('Session ' + str(session))
    #plt.legend()
    #plt.xlabel('Frame seconds')
    #plt.ylabel('Value')
    #plt.show()

deltas = []
cumsums = []
deltacumsum = []
    
#go back to df_interp, session by session, and make the frames_seconds be the same for all participants (equal to the max length of the frames_seconds array)
#and then extrapolate the values as per the last value
sessions = df_interp['session'].unique()
for session in sessions:
    df_session = df_interp[df_interp['session']==session]
    
    #get max length of frames_seconds
    max_length = max([len(x) for x in df_session['frame_seconds'].values])
    max_time = max([max(x) for x in df_session['frame_seconds'].values])
    min_time = min([min(x) for x in df_session['frame_seconds'].values])
   
    #get all participants
    participants = df_session['participant_number'].unique()
    #per participant, get all frame_seconds
    for participant in participants:
        old_frame = df_session[df_session['participant_number']==participant]['frame_seconds'].values[0]
        old_values = df_session[df_session['participant_number']==participant]['value'].values[0]
        #create new frame_seconds array
        new_frame_seconds = np.arange(0.0,max_time+0.1,0.1)[:max_length]
        #print('participant 1 ', max_time, np.array_equal(new_frame_seconds,old_frame))
        #if needed, extend old_values to have the same length as new_frame_seconds, by the last value
        if len(old_frame) < max_length:
            old_values = np.append(old_values,[old_values[-1]]*(max_length-len(old_frame)))
        #replace in df_interp
        df_interp.loc[df_session[df_session['participant_number']==participant].index] = [session,participant,new_frame_seconds,old_values]
        print(old_frame.shape,new_frame_seconds.shape,old_values.shape)

    #add information about the derivative of the values
    #get all participants

    df_session = df_interp[df_interp['session']==session]
    participants = df_session['participant_number'].unique()
    
    #per participant, get all frame_seconds
    for participant in participants:
        print(participant)
        old_frame = df_session[df_session['participant_number']==participant]['frame_seconds'].values[0]
        old_values = df_session[df_session['participant_number']==participant]['value'].values[0]
        #calculate delta between every to consequentent values
        delta_values = np.diff(old_values)
        deltas.append(delta_values)
        #calculate cumsum
        cumsum_values = np.cumsum(old_values)
        cumsums.append(cumsum_values)
        #calculate values with median filter of size 101 (10 seconds)
        median_values = medfilt(old_values,101)
        deltacumsum.append(median_values)

    
#add to df_interp
df_interp['cumsum'] = cumsums
df_interp['derivative'] = deltas
df_interp['medianfilt'] = deltacumsum



#save df_interp
df_interp.to_pickle('df_interpolated1dec.pkl')
#to csv
df_interp.to_csv('df_interpolated1dec.csv')



In [ ]:
df_interp

In [ ]:
#now, the same for values

datapoints = []
value_range_pp = []
value_avg_pp = []
value_std_pp = []

for ind, row in df_interp.iterrows():

    values = df_interp['value'][ind]
    #see range of values
    value_range_pp.append(max(values)-min(values))
    value_avg_pp.append(np.mean(values))
    value_std_pp.append(np.std(values))
    datapoints.extend(values)

#plot just sessions 41 an 38, as a figure with 4 subplots
import tikzplotlib

sns.set(style="ticks", font_scale=2.5)
#import itertools
plt.rcParams["font.family"] = "Times New Roman"


colors0 = ['royalblue']#,'mediumblue','cornflowerblue']
colors1 = ['indianred']#,'firebrick']
colors2 = ['darkgreen']#,'forestgreen','limegreen']
colors3 = ['darkorange']#,'goldenrod','gold']

labs = ['Participant A','Participant B']
fig, ax = plt.subplots(figsize = (15,7))  
#plot distribution of values, with density
#sns.distplot(datapoints, hist=True, color = colors0[0])
#make histogram with bins in np.arange(-7,7,1)
ax.hist(datapoints, bins=[-7,-6,-5,-4,-3,-2,-1,0,1,2,3,4,5,6,7,8],density=True, color = colors0[0])
#plt.title('Distribution of values')
ax.set_xlabel('Interpersonal Perception Ratings')
ax.set_ylabel('Density')
#set ticks
ax.set_xticks(np.arange(-7,8,1))

#plt.show()

#tq_temp['condition'] == 'datadriven']
#    plt.boxplot(df_postq_tempdd[col], labels = ['DataL'], positions = [ind*2+0.9],  widths = 0.6, patch_artist=True, boxprops=dict(facecolor=colors1[0], color=colors1[0]), medianprops=dict(color='black'), whiskerprops=dict(color='black'), capprops=dict(color='black'))
plt.tight_layout()
sns.despine()
plt.show()
#save with high quality in png
fig.savefig('results1.png', dpi=500)

#tikzplotlib.save("results2.tex")

#for ind,col in enumerate(cols_oi):
#    df_postq_tempn = df_postq_temp.loc[df_postq_temp['condition'] == 'naive']
#    plt.boxplot(df_postq_tempn[col], labels = ['NaïveL'], positions = [ind*2 + 0.1],  widths = 0.6, patch_artist=True, boxprops=dict(facecolor=colors0[0], color=colors0[0]), medianprops=dict(color='black'), whiskerprops=dict(color='black'), capprops=dict(color='black'))
#    df_postq_tempdd = df_postq_temp.loc[df_pos



# Mid Survey Data

In [ ]:
#imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from scipy.interpolate import interp1d
#import mse
from sklearn.metrics import mean_squared_error


In [ ]:
#import the df_interpolated
df_interp = pd.read_pickle('df_interpolated1dec.pkl')
#import disagreement
df_disagreement = pd.read_pickle('df_disagreement1dec.pkl')

In [ ]:
#now, import survey data
df_survey = pd.read_csv('../[Study Data]/survey_data/midq.csv')
#fill in the missing values with string 'NAN'
df_survey = df_survey.fillna('NAN')

In [ ]:
#go through the participants included in the df_interp

#start new, clean participant df
df_midq = pd.DataFrame()
pids = []
sessions = []
ages = []
genders = []
nationalities = []
races = []
proficiencies = []
extraversions = []
agreeablenesses = []
conscientiousnesses = []
neuroticisms = []
opennesses = []
religiousnesses = []
politicals = []
agreeables_general = [] #7-point likert
relationalcs = [] #figure out how to calculate
likingscales = []


#dictionary from likert scale to numerical value
likert_dict = {'Strongly disagree':1,'Somewhat disagree':2,'Neither agree nor disagree':3,'Somewhat agree':4,'Strongly agree':5,'NAN':np.nan}
likert_dict7 = {'Strongly Disagree':1,'Disagree':2,'Somewhat disagree':3,'Neither agree nor disagree':4,'Somewhat agree':5,'Agree':6,'Strongry Agree':7,'NAN':np.nan}
religious_dict = {'Not at all religious':1,'2':2,'3':3,'Somewhat Religious':4,'5':5,'6':6,'Very Religious':7,'NAN':np.nan}
political_dict = {'Very Conservative':1,'10':2,'11':3,'Central':4,'13':5,'14':6,'Very Liberal':7,'NAN':np.nan}

#replace value '6152be1021c9ab590015a3e3'in df_survey['Q21'] with 29
df_survey['Q21'] = df_survey['Q21'].replace('6152be1021c9ab590015a3e3','29')
df_survey['Q21'] = df_survey['Q21'].replace('5c6efe3a04b76500015f9756','44')


participants = df_interp['participant_number'].unique()
for participant in participants:
    #select the survey data for this participant
    session = np.ceil((participant-1)/2)


    
    if str(participant) in df_survey['Q21'].values:
        print(participant)
        pids.append(participant)
        df_p = df_survey[df_survey['Q21']==str(participant)]
        sessions.append(session)
        ages.append(df_p['Q1'].values[0])
        genders.append(df_p['Q2'].values[0])
        nationalities.append(df_p['Q3'].values[0])
        races.append(df_p['Q4'].values[0])
        proficiencies.append(df_p['Q5'].values[0])
        extra1 = - likert_dict[df_p['Q6_1'].values[0]] + 6
        extra2 = likert_dict[df_p['Q6_6'].values[0]]
        agree1 = likert_dict[df_p['Q6_2'].values[0]]
        agree2 = - likert_dict[df_p['Q6_7'].values[0]] + 6
        conscious1 = - likert_dict[df_p['Q6_3'].values[0]] + 6
        conscious2 = likert_dict[df_p['Q6_8'].values[0]]
        neuro1 = - likert_dict[df_p['Q6_4'].values[0]] + 6
        neuro2 = likert_dict[df_p['Q6_9'].values[0]]
        open1 = - likert_dict[df_p['Q6_5'].values[0]] + 6
        open2 = likert_dict[df_p['Q6_10'].values[0]]
        extraversions.append(extra1 + extra2)
        agreeablenesses.append(agree1 + agree2)
        conscientiousnesses.append(conscious1 + conscious2)
        neuroticisms.append(neuro1 + neuro2)
        opennesses.append(open1 + open2)
        religiousnesses.append(religious_dict[df_p['Q7_1'].values[0]])
        politicals.append(political_dict[df_p['Q22_1'].values[0]])
        agreeables_general.append(df_p['Q14_1'].values[0])
        relation1 = likert_dict7[df_p['Relational Commun_1'].values[0]] #receptivity/social orientation - He/She tried to establish good relations
        relation2 = likert_dict7[df_p['Relational Commun_2'].values[0]] #receptivity - He/She emphasized agreement between us.
        relation3 = likert_dict7[df_p['Relational Commun_3'].values[0]] #receptivity - He/She was willing to listen to me
        relation4 = likert_dict7[df_p['Relational Commun_4'].values[0]] #receptivity - He/She was sincere
        relation5 = likert_dict7[df_p['Relational Commun_5'].values[0]] #receptivity - He/She wanted to appear reasonable
        relation6 = 8 - likert_dict7[df_p['Relational Commun_6'].values[0]] #arousal - He/She was very unemotional
        relation7 = 8 - likert_dict7[df_p['Relational Commun_7'].values[0]] #nonimmediacy - He/She created a sense of distance between us.
        relation8 = 8 - likert_dict7[df_p['Relational Commun_8'].values[0]] #dominance - He/She seemed to have higher status than me
        relation9 = 8 - likert_dict7[df_p['Relational Commun_9'].values[0]] #intimacy - He/She didn't care if I liked him/her
        relation10 = likert_dict7[df_p['Relational Commun_10'].values[0]] #receptivity . He/She wanted me to trust him/her
        relation11 = likert_dict7[df_p['Relational Commun_11'].values[0]] #intimacy .  He/she expressed attraction towards me
        relation12 = likert_dict7[df_p['Relational Commun_12'].values[0]] #intimacy - He/she made the conversation seem intimate
        relation13 = likert_dict7[df_p['Relational Commun_13'].values[0]] #dominance ? - He/she considered us equals.
        relation14 = likert_dict7[df_p['Relational Commun_14'].values[0]]#composure/emotional non arousal - He/she felt very relaxed talking with me
        relation15 = 8 - likert_dict7[df_p['Relational Commun_15'].values[0]] #arousal/Intimacy/involvement - He/she acted bored by the conversation
        relation16 = 8 - likert_dict7[df_p['Relational Commun_16'].values[0]] #nonimmediacy/intimacy/depth - He/she made the conversation seem superficial.
        relationalcs.append(np.nan) #TODO TODO TODO TODO TODO TODO
        liking1 = likert_dict7[df_p['liking scale_1'].values[0]] #liking
        liking2 = likert_dict7[df_p['liking scale_2'].values[0]] #liking
        liking3 = likert_dict7[df_p['liking scale_3'].values[0]] #liking
        liking4 = likert_dict7[df_p['liking scale_4'].values[0]] #liking
        liking5 = likert_dict7[df_p['liking scale_5'].values[0]] #liking
        liking6 = likert_dict7[df_p['liking scale_6'].values[0]] #liking
        liking7 = likert_dict7[df_p['liking scale_7'].values[0]] #liking
        likingscales.append((liking1 + liking2 + liking3 + liking4 + liking5 + liking6 + liking7)/7)


df_midq['participant'] = pids 
df_midq['session'] = sessions
df_midq['age'] = ages
df_midq['gender'] = genders
df_midq['nationality'] = nationalities
df_midq['race'] = races
df_midq['proficiency'] = proficiencies
df_midq['extraversion'] = extraversions
df_midq['agreeableness'] = agreeablenesses
df_midq['conscientiousness'] = conscientiousnesses
df_midq['neuroticism'] = neuroticisms
df_midq['openness'] = opennesses
df_midq['religiousness'] = religiousnesses
df_midq['political'] = politicals
df_midq['agreeable_general'] = agreeables_general
df_midq['relationalc'] = relationalcs
df_midq['likingscale'] = likingscales
    
#save
df_midq.to_pickle('midq_processed.pkl')
#save to csv
df_midq.to_csv('midq_processed.csv', index=False)

df_midq


# Disagreement from CORAE

In [ ]:
#imports
import pandas as pd
from sklearn.metrics import cohen_kappa_score

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from scipy.interpolate import interp1d
#import mse
from sklearn.metrics import mean_squared_error
import scipy.stats as stats
import itertools
from scipy.stats import pearsonr
from scipy.integrate import trapz
import tikzplotlib

#for plots
import matplotlib as mpl

sns.set(style="whitegrid", font_scale=1.2)#, font='Arial')
#sns.set_palette("husl")
#plt.rcParams.update({'font.size': '50'})
#plt.rcParams['font.size'] = '50'
#mpl.rcParams['font.family'] = 'Arial'

from matplotlib import cm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from termcolor import colored


In [ ]:
#open df_interp
df_interp = pd.read_pickle('df_interpolated1dec.pkl')
#open midq
df_midq = pd.read_pickle('midq_processed.pkl')

In [ ]:


#now, we are able to determine disagreement between participants, by comparing the values across a session for the participants in it
#we can do this by calculating the mean squared error between the values of the participants (?) or by calculating the correlation between the values of the participants (?)
#also, we can just calculate the absolute difference and normalize it through time


#first, select session
sessions = df_interp['session'].unique()
#create a dataframe to store this
df_disagreement = pd.DataFrame(columns=['session','mse','corr','abs_diff', 'cohen_kappa','valuesum','interpersonal_list','interpersonal_diff', 'agreeable_general_sum','likingscale_sum',
                                        'extraversion_diff','agreeableness_diff','conscientiousness_diff','neuroticism_diff','openness_diff',
                                        'religiousness_diff','political_diff','agreeable_general_diff','relationalc_diff','likingscale_diff', 'valuesum_diff',
                                        
                                        'extraversion',
                                        'agreeableness','conscientiousness','neuroticism','openness','religiousness','political','agreeable_general','relationalc','likingscale'], index=range(len(sessions)))


for session in sessions:
    df_session = df_interp[df_interp['session']==session]
    print(df_session['participant_number'].values)
    #get values for both participants
    value_list = df_session['value'].values
    x = df_session['frame_seconds'].values
    #derivative
    derivative_list = df_session['derivative'].values
    
    #calculate mean squared error
    mse = mean_squared_error(value_list[0],value_list[1])

    #calculate correlation
    corr = np.corrcoef(value_list[0],value_list[1])[0,1]
    #calculate absolute difference
    abs_diff = np.sum(np.abs(value_list[0]-value_list[1]))/len(value_list[0])
    #calculate cohen kappa score
    cohen_kappa = cohen_kappa_score(value_list[0],value_list[1])
    #calculate interpersonal difference
    #do this by calculating the cumulative sum of values for each participant, and then calculating the difference between the two (save the values in a list)
    valuesum = np.abs(np.sum(value_list[0]) - np.sum(value_list[1]))/len(value_list[0])
    valuesum_list = [np.sum(value_list[0])/len(value_list[0]),np.sum(value_list[1])/len(value_list[0])]
    #area under curve for each participant
    y0 = np.array(value_list[0])
    x0 = np.array(x[0])
    area0 = (np.trapz(y0[y0>0],x0[y0>0]) - np.trapz(y0[y0<0],x0[y0<0]))/len(value_list[0])
    y1 = np.array(value_list[1])
    x1 = np.array(x[1])
    area1 = (np.trapz(y1[y1>0],x1[y1>0]) - np.trapz(y1[y1<0],x1[y1<0]))/len(value_list[0])
    interpersonal_list = [area0,area1]
    interpersonal_diff = np.abs(area0-area1)
    #then, get the midq values for each session
    midqsession = df_midq[df_midq['session']==session]
    #get the values for the midq
    midqvalues = midqsession['extraversion'].values
    #add to df_disagreement as list
    extraversion = midqvalues.tolist()
    extraversion_diff = np.abs(extraversion[0]-extraversion[1])
    #repeat for all other midq values
    midqvalues = midqsession['agreeableness'].values
    agreeableness = midqvalues.tolist()
    agreeableness_diff = np.abs(agreeableness[0]-agreeableness[1])
    midqvalues = midqsession['conscientiousness'].values
    conscientiousness = midqvalues.tolist()
    conscientiousness_diff = np.abs(conscientiousness[0]-conscientiousness[1])
    midqvalues = midqsession['neuroticism'].values
    neuroticism = midqvalues.tolist()
    neuroticism_diff = np.abs(neuroticism[0]-neuroticism[1])
    midqvalues = midqsession['openness'].values
    openness = midqvalues.tolist()
    openness_diff = np.abs(openness[0]-openness[1])
    midqvalues = midqsession['religiousness'].values
    religiousness = midqvalues.tolist()
    religiousness_diff = np.abs(religiousness[0]-religiousness[1])
    midqvalues = midqsession['political'].values
    political = midqvalues.tolist()
    political_diff = np.abs(political[0]-political[1])
    midqvalues = midqsession['agreeable_general'].values
    agreeable_general = midqvalues.tolist()
    agreeable_general_diff = np.abs(float(agreeable_general[0])-float(agreeable_general[1]))
    agreeable_general_sum = float(agreeable_general[0])+float(agreeable_general[1])
    midqvalues = midqsession['relationalc'].values
    relationalc = midqvalues.tolist()
    relationalc_diff = np.abs(float(relationalc[0])-float(relationalc[1]))
    midqvalues = midqsession['likingscale'].values
    likingscale = midqvalues.tolist()
    likingscale_diff = np.abs(float(likingscale[0])-float(likingscale[1]))
    likingscale_sum = np.sum(likingscale)
    
    #add to df_disagreement
    df_disagreement.loc[df_session.index[0]/2] = [session,mse,corr,abs_diff,cohen_kappa,valuesum,interpersonal_list,interpersonal_diff, agreeable_general_sum,likingscale_sum,
                                                  extraversion_diff,agreeableness_diff,conscientiousness_diff,neuroticism_diff,openness_diff,
                                                  religiousness_diff,political_diff,agreeable_general_diff,relationalc_diff,likingscale_diff, valuesum_list,
                                                  extraversion,
                                                  agreeableness,conscientiousness,neuroticism,openness,religiousness,political,agreeable_general,relationalc,likingscale]


#save df_disagreement
df_disagreement.to_pickle('df_disagreement1dec.pkl')
#to csv
df_disagreement.to_csv('df_disagreement1dec.csv')

df_disagreement
    




In [ ]:
#now, plot 2 plots per session: in one, the values of the two participants, and in the other the cumulative sum of the values of each participant
#sessions
sessions = df_interp['session'].unique()
for session in sessions:
    df_session = df_interp[df_interp['session']==session]
    #for each participant
    participants = df_session['participant_number'].unique()
    fig, (ax1, ax2, ax3) = plt.subplots(nrows=1, ncols=3,figsize=(22,5))
    for participant in participants:
        df_participant = df_session[df_session['participant_number']==participant]
        #plot the values
        x = np.array(df_participant['frame_seconds'].values[0])
        y = np.array(df_participant['value'].values[0])
        ax1.plot(x,y,label = 'Participant ' + str(participant))
        #now, the derivative for each value of y (delta between last value and new one)
        y_diff = np.array(df_participant['derivative'].values[0])
        print(len(y_diff),len(x))
        ax3.plot(x[1:],y_diff,label = 'Participant ' + str(participant))
        #now the cumulative sum
        y_cumsum = np.array(df_participant['cumsum'].values[0])
        ax2.plot(x,y_cumsum,label = 'Participant ' + str(participant))
        #now the cumulative sum of the derivative
        #y_cumsum_diff = np.array(df_participant['medianfilt'].values[0])
        #ax4.plot(x,y_cumsum_diff,label = 'Participant ' + str(participant))

        #add title and legend
    ax1.set_title('Session ' + str(session))
    ax1.legend()
    ax1.set_xlabel('Frame seconds')
    ax1.set_ylabel('Value')
    ax3.legend()
    ax3.set_xlabel('Frame seconds')
    ax3.set_ylabel('Derivative')
    ax2.set_xlabel('Frame seconds')
    ax2.set_ylabel('Cumulative sum')
    ax2.legend()
    #ax4.set_xlabel('Frame seconds')
    #ax4.set_ylabel('Median filtered values')
    #ax4.legend()
    plt.show()

    

In [ ]:
#plot just sessions 41 an 38, as a figure with 4 subplots
import tikzplotlib

sns.set(style="ticks", font_scale=2)
#import itertools
plt.rcParams["font.family"] = "Times New Roman"


colors0 = ['royalblue']#,'mediumblue','cornflowerblue']
colors1 = ['indianred']#,'firebrick']
colors2 = ['darkgreen']#,'forestgreen','limegreen']
colors3 = ['darkorange']#,'goldenrod','gold']

labs = ['Participant A','Participant B']
fig, ((ax1,ax2),(ax3,ax4)) = plt.subplots(2,2,figsize = (20,9))  

sessions = [41,38]
for session in sessions:
    df_session = df_interp[df_interp['session']==session]
    #for each participant
    participants = df_session['participant_number'].unique()
    if session == 41:
        x = np.array(df_session['frame_seconds'].values[0])
        y0 = np.array(df_session['value'].values[0])
        ax1.plot(x,y0,label = labs[0], color = colors0[0],linewidth=3)
        y1 = np.array(df_session['value'].values[1])
        ax1.plot(x,y1,label = labs[1], color = colors1[0],linewidth=3)
        #now, plot the cumulative sum
        y_cumsum0 = np.array(df_session['cumsum'].values[0])
        ax3.plot(x,y_cumsum0,label = labs[0], color = colors0[0],linewidth=3)
        y_cumsum1 = np.array(df_session['cumsum'].values[1])
        ax3.plot(x,y_cumsum1,label = labs[1], color = colors1[0],linewidth=3)
    else:
        x = np.array(df_session['frame_seconds'].values[0])
        y0 = np.array(df_session['value'].values[0])
        ax2.plot(x,y0,label = labs[0], color = colors2[0],linewidth=3)
        y1 = np.array(df_session['value'].values[1])
        ax2.plot(x,y1,label = labs[1], color = colors3[0],linewidth=3)
        #now, plot the cumulative sum
        y_cumsum0 = np.array(df_session['cumsum'].values[0])
        ax4.plot(x,y_cumsum0,label = labs[0], color = colors2[0],linewidth=3)
        y_cumsum1 = np.array(df_session['cumsum'].values[1])
        ax4.plot(x,y_cumsum1,label = labs[1], color = colors3[0],linewidth=3)

#ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Interpersonal Rating')
#set scale from -7 to 7
ax1.set_ylim([-7,8])
#set yticks
ax1.set_yticks([-7,-3,0,3,7])
#add horizontal dashed line at 0
ax1.axhline(y=0, color='k', linestyle='--',linewidth=1, alpha = 0.5)
#set minimum x value to 0, but no upper bound
ax1.set_xlim([0,None])
#ax2.set_xlabel('Time (s)')
#ax2.set_ylabel('Interpersonal Rating')
#set scale from -7 to 7
ax2.set_ylim([-7,8])
#set yticks
ax2.set_yticks([-7,-3,0,3,7])
#add horizontal dashed line at 0
ax2.axhline(y=0, color='k', linestyle='--',linewidth=1, alpha = 0.5)
#set minimum x value to 0, but no upper bound
ax2.set_xlim([0,None])
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('Cumulative IR')
#set minimum x value to 0, but no upper bound
ax3.set_xlim([0,None])
ax4.set_xlabel('Time (s)')
#ax4.set_ylabel('Cumulative IR')
#set minimum x value to 0, but no upper bound
ax4.set_xlim([0,None])
#put legend in lower left corner
#ax1.legend(loc = 'lower left')
ax3.legend()
ax4.legend()

#tq_temp['condition'] == 'datadriven']
#    plt.boxplot(df_postq_tempdd[col], labels = ['DataL'], positions = [ind*2+0.9],  widths = 0.6, patch_artist=True, boxprops=dict(facecolor=colors1[0], color=colors1[0]), medianprops=dict(color='black'), whiskerprops=dict(color='black'), capprops=dict(color='black'))
plt.tight_layout()
#sns.despine()
plt.show()
#save with high quality in png
fig.savefig('../figures/results2.png', dpi=300)

#tikzplotlib.save("results2.tex")

#for ind,col in enumerate(cols_oi):
#    df_postq_tempn = df_postq_temp.loc[df_postq_temp['condition'] == 'naive']
#    plt.boxplot(df_postq_tempn[col], labels = ['NaïveL'], positions = [ind*2 + 0.1],  widths = 0.6, patch_artist=True, boxprops=dict(facecolor=colors0[0], color=colors0[0]), medianprops=dict(color='black'), whiskerprops=dict(color='black'), capprops=dict(color='black'))
#    df_postq_tempdd = df_postq_temp.loc[df_pos

In [ ]:
#plot just sessions 41 an 38, as a figure with 4 subplots
import tikzplotlib

sns.set(style="ticks", font_scale=2)
#import itertools
plt.rcParams["font.family"] = "Times New Roman"


colors0 = ['royalblue']#,'mediumblue','cornflowerblue']
colors1 = ['indianred']#,'firebrick']
colors2 = ['darkgreen']#,'forestgreen','limegreen']
colors3 = ['darkorange']#,'goldenrod','gold']

labs = ['Participant A','Participant B']

sessions = df_interp['session'].unique()
for session in sessions:
    df_session = df_interp[df_interp['session']==session]
    #for each participant
    participants = df_session['participant_number'].unique()
    fig, (ax1,ax2) = plt.subplots(1,2,figsize = (20,7))  

    x = np.array(df_session['frame_seconds'].values[0])
    y0 = np.array(df_session['value'].values[0])
    ax1.plot(x,y0,label = labs[0], color = colors0[0],linewidth=3)
    y1 = np.array(df_session['value'].values[1])
    ax1.plot(x,y1,label = labs[1], color = colors1[0],linewidth=3)
    #now, plot the cumulative sum
    y_cumsum0 = np.array(df_session['cumsum'].values[0])
    ax2.plot(x,y_cumsum0,label = labs[0], color = colors0[0],alpha = 0.6, linewidth=3)
    y_cumsum1 = np.array(df_session['cumsum'].values[1])
    ax2.plot(x,y_cumsum1,label = labs[1], color = colors1[0],alpha = 0.6, linewidth=3)

    slope, intercept, r_value, p_value, std_err = stats.linregress(x,y_cumsum0)
    line = slope*x+intercept
    ax2.plot(x,line,'--',color = colors2[0],linewidth=3,label = 'R\u00b2 : ' + str(round(r_value, 4)))
    slope, intercept, r_value, p_value, std_err = stats.linregress(x,y_cumsum1)
    line = slope*x+intercept
    ax2.plot(x,line,'--',color = colors3[0],linewidth=3, label = 'R\u00b2 : ' + str(round(r_value, 4)))

    #ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Interpersonal Rating')
    #set scale from -7 to 7
    ax1.set_ylim([-7,8])
    #set yticks
    ax1.set_yticks([-7,-3,0,3,7])
    #add horizontal dashed line at 0
    ax1.axhline(y=0, color='k', linestyle='--',linewidth=1, alpha = 0.5)
    #set minimum x value to 0, but no upper bound
    ax1.set_xlim([0,None])

    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Cumulative IR')
    #set minimum x value to 0, but no upper bound
    ax2.set_xlim([0,None])
    ax2.legend()

    #tq_temp['condition'] == 'datadriven']
    #    plt.boxplot(df_postq_tempdd[col], labels = ['DataL'], positions = [ind*2+0.9],  widths = 0.6, patch_artist=True, boxprops=dict(facecolor=colors1[0], color=colors1[0]), medianprops=dict(color='black'), whiskerprops=dict(color='black'), capprops=dict(color='black'))
    plt.tight_layout()
    #sns.despine()
    plt.show()  
    #save with high quality in png
    fig.savefig('../figures/supp' + str(session) + '.png', dpi=300)

#tikzplotlib.save("results2.tex")

#for ind,col in enumerate(cols_oi):
#    df_postq_tempn = df_postq_temp.loc[df_postq_temp['condition'] == 'naive']
#    plt.boxplot(df_postq_tempn[col], labels = ['NaïveL'], positions = [ind*2 + 0.1],  widths = 0.6, patch_artist=True, boxprops=dict(facecolor=colors0[0], color=colors0[0]), medianprops=dict(color='black'), whiskerprops=dict(color='black'), capprops=dict(color='black'))
#    df_postq_tempdd = df_postq_temp.loc[df_pos

In [ ]:
#now, fit a linear regression to the cumulative sum of the values of each participant, and plot the values of the two participants and the regression line
#sessions
slopes = []
r_values = []
p_values = []
participant_list = []

sessions = df_interp['session'].unique()
for session in sessions:
    df_session = df_interp[df_interp['session']==session]
    #for each participant
    participants = df_session['participant_number'].unique()
    fig, ax = plt.subplots(figsize=(15,5))
    for participant in participants:
        df_participant = df_session[df_session['participant_number']==participant]
        #plot the values
        x = np.array(df_participant['frame_seconds'].values[0])
        y = np.array(df_participant['cumsum'].values[0])
        ax.plot(x,y,label = 'Participant ' + str(participant))
        #now, fit a linear regression
        slope, intercept, r_value, p_value, std_err = stats.linregress(x,y)
        line = slope*x+intercept
        print(len(line),len(x))
        slopes.append(slope)
        r_values.append(r_value)
        p_values.append(p_value)
        participant_list.append(participant)
        ax.plot(x,line,label = str(r_value))

    #add title and legend
    ax.set_title('Session ' + str(session))
    ax.legend()
    ax.set_xlabel('Frame seconds')
    ax.set_ylabel('Cumulative sum')
    plt.show()


#add to df_disagreement
df_disagreement['slopes'] = [i for i in np.array(slopes).reshape(int(len(slopes)/2), 2)]
df_disagreement['r_values'] = [i for i in np.array(r_values).reshape(int(len(slopes)/2), 2)]
df_disagreement['p_values'] = [i for i in np.array(p_values).reshape(int(len(slopes)/2), 2)]

df_disagreement['slope_diff'] = np.abs(df_disagreement['slopes'].apply(lambda x: x[0])-df_disagreement['slopes'].apply(lambda x: x[1]))

#now, add slopes to df_midq, according to the participant number
for participant in participant_list:
    df_midq.loc[df_midq['participant']==participant,'slope'] = slopes[participant_list.index(participant)]
    #add also the values in the df_disagreement['interpersonal_list']
    session = df_midq[df_midq['participant']==participant]['session'].values[0]
    interpersonal_list = df_disagreement[df_disagreement['session']==session]['interpersonal_list'].values

    #check if participant number is even or odd
    if participant%2==0:
        df_midq.loc[df_midq['participant']==participant,'interpersonal'] = interpersonal_list[0][0]
    else:
        df_midq.loc[df_midq['participant']==participant,'interpersonal'] = interpersonal_list[0][1]

#save df_disagreement
df_disagreement.to_pickle('df_disagreement1dec.pkl')
#to csv
df_disagreement.to_csv('df_disagreement1dec.csv')

#save df_midq
df_midq.to_pickle('midq_processed.pkl')
#to csv
df_midq.to_csv('midq_processed.csv')

df_disagreement
    




In [ ]:
df_disagreement.columns

In [ ]:
r_values =np.array([0.95059199, 0.99983174, 0.9036017, 0.9144214, 0.99462086, 0.97762785, 0.99980018, 0.99982248, 
                    0.90478528, -0.6934582, 0.99351582, 0.99894149, 0.99398677, 0.79835396, 0.97484511, 0.99706833,
                      0.47062635, 0.99774366, 0.99213275, 0.99629849, 0.89715595, 0.99479227, 0.99199593, 0.99646349])
print(np.mean(r_values))
print(np.std(r_values))

In [ ]:
#now, plot some of the datapoints for each session, from df_disagreement
cols_oi = ['mse', 'corr', 'abs_diff', 'cohen_kappa', 'valuesum',
       'interpersonal_diff', 'agreeable_general_sum', 'likingscale_sum',
       'extraversion_diff', 'agreeableness_diff', 'conscientiousness_diff',
       'neuroticism_diff', 'openness_diff', 'religiousness_diff',
       'political_diff', 'agreeable_general_diff', 
       'likingscale_diff', 'slope_diff']

#MISSING THE RELATIONALC_DIFF

#sessions
sessions = df_disagreement['session'].unique()
#get a color palette with the number of sessions we have
colors = sns.color_palette("hls", len(cols_oi))
fig,ax = plt.subplots(figsize=(15,5))
#for ind,session in enumerate(sessions):
#    #plot each column in the same column
#    df_session = df_disagreement[df_disagreement['session']==session]
#    for indx,col in enumerate(cols_oi):
#        sns.scatterplot(x=ind, y = df_session[col],ax=ax,label = col,color=colors[indx])

for indx,col in enumerate(cols_oi):
        sns.lineplot(x=np.arange(len(sessions)), y = df_disagreement[col].values,ax=ax,label = col,color=colors[indx])
    

In [ ]:
#now, use pearson correlation to find which variables better correlate with the slope difference
df_disagreement_numeric = df_disagreement[cols_oi].astype(float)
df_corr = df_disagreement_numeric.corr(method='pearson')
pval = df_disagreement_numeric.corr(method=lambda x, y: pearsonr(x, y)[1]) - np.eye(*df_corr.shape)
p = pval.applymap(lambda x: ''.join(['*' for t in [.05, .01, .001] if x<=t]))
df_corr_pval = df_corr.round(2).astype(str) + p
df_corr_pval

In [ ]:
#plot df_corr as a correlation matrix
fig, ax = plt.subplots(figsize=(15,10))
sns.heatmap(df_corr, annot=True, ax=ax)


In [ ]:
#print participant demographics
#avg age + std
df_midq['age'] = df_midq['age'].astype(float)
print('Average age: ',df_midq['age'].mean(), ' +- ', df_midq['age'].std())
#age range
print('Age range: ',df_midq['age'].min(), ' - ', df_midq['age'].max())
#gender distribution - count occurrences of every value
print( df_midq['gender'].value_counts())
#race distribution
print(df_midq['race'].value_counts())
print(df_midq['proficiency'].value_counts())
df_midq

In [ ]:
cols_midq = ['extraversion', 'agreeableness', 'conscientiousness',
       'neuroticism', 'openness', 'religiousness', 'political',
       'agreeable_general', 'relationalc', 'likingscale', 'interpersonal','slope']

#get correlation matrix
df_midq_numeric = df_midq[cols_midq].astype(float)
df_corr = df_midq_numeric.corr(method='pearson')
pval = df_midq_numeric.corr(method=lambda x, y: pearsonr(x, y)[1]) - np.eye(*df_corr.shape)
p = pval.applymap(lambda x: ''.join(['*' for t in [.05, .01, .001] if x<=t]))
df_corr_pval = df_corr.round(2).astype(str) + p
df_corr_pval

In [ ]:
#plot
fig, ax = plt.subplots(figsize=(10,10))
sns.heatmap(df_corr, annot=True, ax=ax)
